# Parallel DNA Complement Generation

In [1]:
%%writefile parallel_dna_complement_generation.cu
#include <stdio.h>
#include <stdlib.h>
#include <cuda_runtime.h>

#define N 24000000

__global__ void codon_warp_mapping(const char* sequence, int n) {
    int gid = blockIdx.x * blockDim.x + threadIdx.x;

    if (gid < n) {
        volatile int codon_id = gid / 3;

        volatile int warp_id = threadIdx.x / 32;
        volatile int lane_id = threadIdx.x % 32;
    }
}

int main() {
    char* h_sequence = (char*)malloc(N * sizeof(char));

    for (int i = 0; i < N; i++) {
        h_sequence[i] = 'A';
    }

    char* d_sequence;
    cudaMalloc((void**)&d_sequence, N * sizeof(char));

    cudaMemcpy(d_sequence, h_sequence, N * sizeof(char), cudaMemcpyHostToDevice);

    int block_size = 256;
    int grid_size = (N + block_size - 1) / block_size;

    codon_warp_mapping<<<grid_size, block_size>>>(d_sequence, N);

    cudaDeviceSynchronize();

    long long total_codons = N / 3;
    long long total_threads = (long long)grid_size * block_size;
    long long total_warps = total_threads / 32;

    printf("--- Execution and Mapping Analysis ---\n");
    printf("Total DNA Nucleotides: %d\n", N);
    printf("Total number of codons: %lld\n", total_codons);
    printf("Total number of threads: %lld\n", total_threads);
    printf("Total number of warps: %lld\n", total_warps);

    free(h_sequence);
    cudaFree(d_sequence);

    return 0;
}

Writing parallel_dna_complement_generation.cu


In [2]:
!nvcc -arch=sm_75 parallel_dna_complement_generation.cu -o pdnacg

In [3]:
!./pdnacg

--- Execution and Mapping Analysis ---
Total DNA Nucleotides: 24000000
Total number of codons: 8000000
Total number of threads: 24000000
Total number of warps: 750000


To process a flat 1D array, each thread must calculate a unique ID (tid = blockIdx.x * blockDim.x + threadIdx.x). This mapping ensures that no two threads process the same nucleotide.

The array size is exactly 10,000,000. Because we divide this by block sizes like 64 or 256, the total number of threads launched is often slightly larger than 10,000,000 (e.g., for block size 256, total threads = 10,000,128). The if (tid < n) check is strictly necessary; without it, the last few threads in the final block would access unallocated memory, causing a segmentation fault on the GPU.

The kernel uses an if-else chain to determine the complement. In CUDA, threads are executed in groups of 32 - warps. If threads within a warp follow different execution paths (e.g., some hit the A condition, others hit T), this is called warp divergence, and it serializes execution slightly.

Because the DNA is randomly generated, divergence is guaranteed here. However, because the operations are just simple variable assignments, the performance penalty is negligible compared to memory latency.

## Parallel DNA Base Frequency Analysis

In [4]:
%%writefile parallel_dna_base_frequency_analysis.cu
#include <stdio.h>
#include <stdlib.h>
#include <cuda_runtime.h>
#include <time.h>

#define N 20000000

__global__ void count_bases(const char* sequence, int* counts, int n) {
    int tid = blockIdx.x * blockDim.x + threadIdx.x;

    if (tid < n) {
        char base = sequence[tid];

        if (base == 'A') {
            atomicAdd(&counts[0], 1);
        } else if (base == 'C') {
            atomicAdd(&counts[1], 1);
        } else if (base == 'G') {
            atomicAdd(&counts[2], 1);
        } else if (base == 'T') {
            atomicAdd(&counts[3], 1);
        }
    }
}

void run_test(int block_size, const char* d_sequence) {
    int grid_size = (N + block_size - 1) / block_size;

    int h_counts[4] = {0, 0, 0, 0};
    int* d_counts;
    cudaMalloc((void**)&d_counts, 4 * sizeof(int));
    cudaMemcpy(d_counts, h_counts, 4 * sizeof(int), cudaMemcpyHostToDevice);

    cudaEvent_t start, stop;
    cudaEventCreate(&start);
    cudaEventCreate(&stop);

    cudaEventRecord(start);
    count_bases<<<grid_size, block_size>>>(d_sequence, d_counts, N);
    cudaEventRecord(stop);

    cudaEventSynchronize(stop);

    float milliseconds = 0;
    cudaEventElapsedTime(&milliseconds, start, stop);

    cudaMemcpy(h_counts, d_counts, 4 * sizeof(int), cudaMemcpyDeviceToHost);

    int total_sum = h_counts[0] + h_counts[1] + h_counts[2] + h_counts[3];

    printf("--- Block Size: %d ---\n", block_size);
    printf("Number of Blocks: %d\n", grid_size);
    printf("Count of A:       %d\n", h_counts[0]);
    printf("Count of C:       %d\n", h_counts[1]);
    printf("Count of G:       %d\n", h_counts[2]);
    printf("Count of T:       %d\n", h_counts[3]);
    printf("Total Sum:        %d (Expected: %d)\n", total_sum, N);
    printf("Execution Time:   %.3f ms\n\n", milliseconds);

    cudaFree(d_counts);
    cudaEventDestroy(start);
    cudaEventDestroy(stop);
}

int main() {
    srand(time(NULL));

    char* h_sequence = (char*)malloc(N * sizeof(char));

    char nucleotides[] = {'A', 'C', 'G', 'T'};
    for (int i = 0; i < N; i++) {
        h_sequence[i] = nucleotides[rand() % 4];
    }

    char* d_sequence;
    cudaMalloc((void**)&d_sequence, N * sizeof(char));

    cudaMemcpy(d_sequence, h_sequence, N * sizeof(char), cudaMemcpyHostToDevice);

    int block_sizes[] = {64, 128, 256, 512};
    int num_tests = sizeof(block_sizes) / sizeof(block_sizes[0]);

    for (int i = 0; i < num_tests; i++) {
        run_test(block_sizes[i], d_sequence);
    }

    free(h_sequence);
    cudaFree(d_sequence);

    return 0;
}

Writing parallel_dna_base_frequency_analysis.cu


In [5]:
!nvcc -arch=sm_75 parallel_dna_base_frequency_analysis.cu -o pdnabfa

In [6]:
!./pdnabfa

--- Block Size: 64 ---
Number of Blocks: 312500
Count of A:       4999178
Count of C:       5003152
Count of G:       4999010
Count of T:       4998660
Total Sum:        20000000 (Expected: 20000000)
Execution Time:   3.230 ms

--- Block Size: 128 ---
Number of Blocks: 156250
Count of A:       4999178
Count of C:       5003152
Count of G:       4999010
Count of T:       4998660
Total Sum:        20000000 (Expected: 20000000)
Execution Time:   3.131 ms

--- Block Size: 256 ---
Number of Blocks: 78125
Count of A:       4999178
Count of C:       5003152
Count of G:       4999010
Count of T:       4998660
Total Sum:        20000000 (Expected: 20000000)
Execution Time:   3.131 ms

--- Block Size: 512 ---
Number of Blocks: 39063
Count of A:       4999178
Count of C:       5003152
Count of G:       4999010
Count of T:       4998660
Total Sum:        20000000 (Expected: 20000000)
Execution Time:   3.130 ms



Because multiple threads in different blocks execute simultaneously, a standard counts[0]++ read-modify-write operation would result in a "race condition." Multiple threads would read the same initial value, increment it, and overwrite each other's work, leading to massively undercounted totals.

atomicAdd ensures that the read-modify-write step happens as a single, uninterrupted transaction. The GPU hardware locks the specific memory address, performs the addition, and releases the lock.

As the block size increases, the number of blocks proportionally decreases to cover the fixed $N = 20,000,000$.



## Codon and Warp Mapping Analysis

In [7]:
%%writefile codon_and_warp_mapping_analysis.cu
#include <stdio.h>
#include <stdlib.h>
#include <cuda_runtime.h>

#define N 24000000

__global__ void codon_warp_mapping(const char* sequence, int n) {
    int gid = blockIdx.x * blockDim.x + threadIdx.x;

    if (gid < n) {
        volatile int codon_id = gid / 3;

        volatile int warp_id = threadIdx.x / 32;
        volatile int lane_id = threadIdx.x % 32;
    }
}

int main() {
    char* h_sequence = (char*)malloc(N * sizeof(char));

    for (int i = 0; i < N; i++) {
        h_sequence[i] = 'A';
    }

    char* d_sequence;
    cudaMalloc((void**)&d_sequence, N * sizeof(char));

    cudaMemcpy(d_sequence, h_sequence, N * sizeof(char), cudaMemcpyHostToDevice);

    int block_size = 256;
    int grid_size = (N + block_size - 1) / block_size;

    codon_warp_mapping<<<grid_size, block_size>>>(d_sequence, N);

    cudaDeviceSynchronize();

    long long total_codons = N / 3;
    long long total_threads = (long long)grid_size * block_size;
    long long total_warps = total_threads / 32;

    printf("--- Execution and Mapping Analysis ---\n");
    printf("Total DNA Nucleotides: %d\n", N);
    printf("Total number of codons: %lld\n", total_codons);
    printf("Total number of threads: %lld\n", total_threads);
    printf("Total number of warps: %lld\n", total_warps);

    free(h_sequence);
    cudaFree(d_sequence);

    return 0;
}


Writing codon_and_warp_mapping_analysis.cu


In [8]:
!nvcc -arch=sm_75 codon_and_warp_mapping_analysis.cu -o cwma

In [9]:
!./cwma

--- Execution and Mapping Analysis ---
Total DNA Nucleotides: 24000000
Total number of codons: 8000000
Total number of threads: 24000000
Total number of warps: 750000


hardware strictly executes in warps (groups of 32 threads). Biological data in this scenario is grouped into codons (groups of 3 nucleotides).

The biological concept (Codon 10) is split physically across two different hardware execution units (Warp 0 and Warp 1).

A better architecture is usually 1 thread = 1 codon